# Kaggle 40M Gated Delta Trainer (Variant E, 100k)Strict Variant-E notebook for non-transformer piano continuation research.Pipeline:1. Setup and dependency checks.2. 40M GDN config + 100k dataset targets.3. Manifest build/discovery.4. Architecture preflight (real GDN, param budget, forward sanity).5. Dry run first, then optional training run.

In [ ]:
from pathlib import Pathimport importlib.utilimport osimport subprocessimport sysREQUIRE_REAL_GDN = TrueREQUIRED = {'miditok':'miditok','pretty_midi':'pretty_midi','ncps':'ncps','safetensors':'safetensors','matplotlib':'matplotlib','huggingface_hub':'huggingface_hub'}missing = [spec for name, spec in REQUIRED.items() if importlib.util.find_spec(name) is None]if missing:    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', *missing])def find_project_root() -> Path:    anchor = Path('training/ablation_runner.py')    probes = [Path.cwd().resolve(), Path('/kaggle/working'), Path('/kaggle/input'), Path('/kaggle/working/piano_midi_model'), Path('/kaggle/input/piano_midi_model')]    for probe in probes:        if not probe.exists():            continue        for parent in [probe, *probe.parents]:            if (parent / anchor).exists() and (parent / 'training' / '__init__.py').exists():                return parent    raise FileNotFoundError('Could not locate project root containing training/ablation_runner.py')PROJECT_ROOT = find_project_root()if str(PROJECT_ROOT) not in sys.path:    sys.path.insert(0, str(PROJECT_ROOT))os.environ['PYTHONPATH'] = str(PROJECT_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')OUTPUT_BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else PROJECT_ROOTfrom model.blocks.gdn_block import try_import_fla, GDN_AVAILABLEif not GDN_AVAILABLE:    try:        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', 'flash-linear-attention==0.7.0'])    except Exception as exc:        print(f'flash-linear-attention install failed: {exc}')    _ = try_import_fla()if REQUIRE_REAL_GDN and not GDN_AVAILABLE:    raise RuntimeError('Real GDN kernels are unavailable. Notebook blocks fallback GDN by default.')print('PROJECT_ROOT:', PROJECT_ROOT)print('OUTPUT_BASE:', OUTPUT_BASE)print('GDN_AVAILABLE:', GDN_AVAILABLE)

In [ ]:
import jsonimport numpy as npimport torchMAX_PIECES = 100_000EPOCHS = 20SEED = 42SEED_LENGTH = 512CONTINUATION_LENGTH = 1536MAX_SEQUENCE_LENGTH = SEED_LENGTH + CONTINUATION_LENGTHTARGET_EFFECTIVE_BATCH = 32BATCH_PER_GPU = 2LEARNING_RATE = 2e-4WARMUP_RATIO = 0.1WARMUP_STEPS = 0MIN_LR_RATIO = 0.1WEIGHT_DECAY = 0.01LABEL_SMOOTHING = 0.1MAX_GRAD_NORM = 1.0LOG_EVERY_N_STEPS = 20SAVE_EVERY_N_EPOCHS = 5KEEP_EVERY_N_EPOCHS = 10MAX_CHECKPOINTS = 8AUTO_RESUME = TrueRESUME_FROM_CHECKPOINT = ''RESUME_MODE = 'remaining'ALLOW_FALLBACK_GDN = FalseALLOW_GDN_DATA_PARALLEL = FalseRUN_TRAINING = FalsePROFILE = {'d_model': 768, 'n_layers': 13, 'attention_every_n_layers': 2, 'gdn_inner_ratio': 0.5, 'gdn_num_heads': 4, 'gqa_num_heads': 8, 'gqa_groups': 4}PRETOKENIZED_ROOT_CANDIDATES = [PROJECT_ROOT / 'processed', Path('/kaggle/input/godzilla-tokenized-100k/tokenized'), Path('/kaggle/input/godzilla-tokenized-100k'), Path('/kaggle/input/godzilla-piano-tokenized-100k/tokenized'), Path('/kaggle/input/godzilla-piano-tokenized-100k'), Path('/kaggle/input/godzilla-tokenized-15k/tokenized'), Path('/kaggle/input/godzilla-tokenized-15k')]GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 1BATCH_SIZE = max(1, GPU_COUNT * BATCH_PER_GPU) if GPU_COUNT > 1 else 1GRAD_ACCUM_STEPS = max(1, int(np.ceil(float(TARGET_EFFECTIVE_BATCH) / float(BATCH_SIZE))))NUM_WORKERS = max(0, int(GPU_COUNT) * 2)ENABLE_DATA_PARALLEL_E = GPU_COUNT > 1run_tag = f'sub100m_e_40m_{MAX_PIECES // 1000}k'OUTPUT_DIR = OUTPUT_BASE / 'outputs' / run_tagPRETOKENIZED_MANIFEST = OUTPUT_DIR / 'processed_pretokenized' / 'manifest.json'RESULT_JSON = OUTPUT_DIR / 'variant_e_40m_result.json'print('Batch:', BATCH_SIZE, 'GradAccum:', GRAD_ACCUM_STEPS)print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
import torchfrom data.tokenizer_custom import CustomDeltaTokenizerfrom model.variant_e import VariantEConfig, VariantEModelfrom training.ablation_runner import _variant_backend_statusdef discover_npz_parent(path: Path):    if not path.exists():        return None    hit = next(path.rglob('*.npz'), None)    return hit.parent if hit is not None else Nonedef find_npz_root(candidates) -> Path:    for c in candidates:        p = discover_npz_parent(Path(c))        if p is not None:            return p    kaggle_input = Path('/kaggle/input')    if kaggle_input.exists():        for folder in kaggle_input.iterdir():            if folder.is_dir():                p = discover_npz_parent(folder)                if p is not None:                    return p    raise FileNotFoundError('Could not locate tokenized NPZ files.')def build_manifest(npz_root: Path, manifest_path: Path) -> int:    npz_files = sorted(npz_root.rglob('*.npz'))    rows = []    for npz_path in npz_files:        try:            with np.load(npz_path, allow_pickle=False) as pack:                if 'tokens' not in pack.files:                    continue                rows.append({'md5': npz_path.stem, 'npz_path': str(npz_path), 'length': int(np.asarray(pack['tokens']).shape[0]), 'source_path': str(npz_path.parent)})        except Exception:            continue    manifest_path.parent.mkdir(parents=True, exist_ok=True)    manifest_path.write_text(json.dumps(rows, indent=2), encoding='utf-8')    return len(rows)OUTPUT_DIR.mkdir(parents=True, exist_ok=True)PRETOKENIZED_ROOT = find_npz_root(PRETOKENIZED_ROOT_CANDIDATES)count = build_manifest(PRETOKENIZED_ROOT, PRETOKENIZED_MANIFEST) if not PRETOKENIZED_MANIFEST.exists() else len(json.loads(PRETOKENIZED_MANIFEST.read_text(encoding='utf-8')))print('Manifest entries:', count)tokenizer = CustomDeltaTokenizer(include_special_tokens=False)event_size = int(getattr(tokenizer, 'event_size', 1))if event_size != 4:    raise RuntimeError(f'Expected event_size=4, got {event_size}')if (SEED_LENGTH % event_size) != 0 or (CONTINUATION_LENGTH % event_size) != 0:    raise RuntimeError('Seed/continuation lengths must be divisible by event_size.')d_model = int(PROFILE['d_model'])gdn_inner_dim = max(128, int(round(float(d_model) * float(PROFILE['gdn_inner_ratio']))))model = VariantEModel(VariantEConfig(vocab_size=int(tokenizer.vocab_size), d_model=d_model, n_layers=int(PROFILE['n_layers']), max_sequence_length=int(MAX_SEQUENCE_LENGTH), gdn_inner_dim=int(gdn_inner_dim), gdn_num_heads=int(PROFILE['gdn_num_heads']), gqa_num_heads=int(PROFILE['gqa_num_heads']), gqa_groups=int(PROFILE['gqa_groups']), attention_every_n_layers=int(PROFILE['attention_every_n_layers'])))params = int(sum(p.numel() for p in model.parameters()))if not (38_000_000 <= params <= 42_000_000):    raise RuntimeError(f'Expected ~40M params, got {params:,}')backend = _variant_backend_status(model)if REQUIRE_REAL_GDN and bool(backend.get('gdn_using_fallback', False)):    raise RuntimeError('Fallback GDN detected, but real GDN is required.')device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')model = model.to(device).eval()probe_len = min(256, int(MAX_SEQUENCE_LENGTH))probe_tokens = torch.randint(0, int(tokenizer.vocab_size), (1, probe_len), dtype=torch.long, device=device)probe_onsets = torch.arange(probe_len, dtype=torch.float32, device=device).unsqueeze(0) * 0.1with torch.no_grad():    logits, _ = model(token_ids=probe_tokens, onset_times=probe_onsets, memory=None, return_memory=True, position_offset=0)if tuple(logits.shape) != (1, probe_len, int(tokenizer.vocab_size)):    raise RuntimeError(f'Unexpected logits shape: {tuple(logits.shape)}')print('Architecture preflight passed.')print(f'Params: {params:,} ({params / 1e6:.2f}M)')print('Backend:', backend)

In [ ]:
from dataclasses import replacefrom pathlib import Pathimport jsonfrom training.sub100m_unified import UnifiedSub100mConfig, run_unified_sub100mcfg = UnifiedSub100mConfig(variant='e', output_dir=str(OUTPUT_DIR), pretokenized_manifest=str(PRETOKENIZED_MANIFEST), pretokenized_root=str(PRETOKENIZED_ROOT), max_pieces=int(MAX_PIECES), seed=int(SEED), seed_length=int(SEED_LENGTH), continuation_length=int(CONTINUATION_LENGTH), max_sequence_length=int(MAX_SEQUENCE_LENGTH), epochs=int(EPOCHS), batch_size=int(BATCH_SIZE), grad_accumulation_steps=int(GRAD_ACCUM_STEPS), learning_rate=float(LEARNING_RATE), warmup_ratio=float(WARMUP_RATIO), warmup_steps=int(WARMUP_STEPS), min_lr_ratio=float(MIN_LR_RATIO), weight_decay=float(WEIGHT_DECAY), label_smoothing=float(LABEL_SMOOTHING), max_grad_norm=float(MAX_GRAD_NORM), num_workers=int(NUM_WORKERS), log_every_n_steps=int(LOG_EVERY_N_STEPS), save_every_n_epochs=int(SAVE_EVERY_N_EPOCHS), keep_every_n_epochs=int(KEEP_EVERY_N_EPOCHS), max_checkpoints=int(MAX_CHECKPOINTS), auto_resume=bool(AUTO_RESUME), resume_from_checkpoint=str(RESUME_FROM_CHECKPOINT), resume_mode=str(RESUME_MODE), enable_data_parallel_c=False, enable_data_parallel_e=bool(ENABLE_DATA_PARALLEL_E), allow_gdn_data_parallel=bool(ALLOW_GDN_DATA_PARALLEL), allow_fallback_gdn=bool(ALLOW_FALLBACK_GDN), dry_run=True)dry_report = run_unified_sub100m(cfg)print('Dry run result:', dry_report['result_path'])print('Dry run params:', dry_report['params'])if bool(RUN_TRAINING):    final_report = run_unified_sub100m(replace(cfg, dry_run=False))    FINAL_RESULT_PATH = Path(final_report['result_path'])    print('Training complete:', FINAL_RESULT_PATH)else:    FINAL_RESULT_PATH = Path(dry_report['result_path'])    print('RUN_TRAINING=False, only dry run executed.')if FINAL_RESULT_PATH.exists():    payload = json.loads(FINAL_RESULT_PATH.read_text(encoding='utf-8'))    print('Profile:', payload.get('profile'))    print('Backend status:', payload.get('backend_status', {}))    print('Training profile:', payload.get('training_profile', {}))

## Optional Next Architecture CandidateA strong follow-up is a dual-rate GDN:- Event-rate local GDN stream for detail.- Downsampled phrase-rate GDN stream for long structure.- Cross-gating from phrase stream into local updates.This keeps the model mostly state-space/recurrent while improving phrase-level coherence.